# Surrogate tree global
**tabular › explain_regression** · notebook `global_surrogate`

**Kapan dipakai:** Menjelaskan model kotak-hitam dengan pohon keputusan dangkal berisi aturan yang bisa dibaca dan ditulis di laporan.

**Jebakan umum:** Hanya perkiraan. Periksa fidelity (R² terhadap prediksi model, bukan terhadap label); fidelity rendah berarti aturannya tidak bisa dipercaya.

## 0. Setup
Import dan lokasi proyek. Tidak perlu diubah.

In [ ]:
import sys, pathlib, warnings
ROOT = next(p for p in [pathlib.Path.cwd().resolve(), *pathlib.Path.cwd().resolve().parents]
            if (p / "src" / "superdb").is_dir())
sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.max_columns", 50)
from superdb.common import *
from superdb.paths import out_dir

## 1. Konfigurasi
Ganti bagian ini saja saat memakai data lomba.

In [ ]:
TRAIN_PATH = ROOT / "data_latihan/tabular/regression/diamonds_train.csv"
TEST_PATH  = ROOT / "data_latihan/tabular/regression/diamonds_test.csv"      # None kalau tidak ada file test
TARGET     = "price"
ID_COL     = 'id'
DROP_COLS  = []
GROUP_COL  = None                      # isi bila ada grup agar tidak bocor
SCORING    = "neg_root_mean_squared_error"
SEED, N_SPLITS, VALID_SIZE = 42, 5, 0.2
MODEL_KEY  = "global_surrogate"
TASK       = "regression"
KIND       = "tree"
OUT = out_dir("tabular", "explain_regression", MODEL_KEY)
set_seed(SEED)

## 2. Baca data

In [ ]:
df = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH) if TEST_PATH else None
print(df.shape, None if test is None else test.shape)
display(df.head(3))

## 3. Fitur & split

In [ ]:
from superdb.tabular import detect_columns, prepare_features, build_pipeline, split_train_valid, cv_score
cols = detect_columns(df, target=TARGET, id_col=ID_COL, drop_cols=DROP_COLS)
X, NUM, CAT = prepare_features(df, cols)
y = df[TARGET].to_numpy(dtype=float)
groups = df[GROUP_COL] if GROUP_COL else None
Xtr, Xva, ytr, yva = split_train_valid(X, y, TASK, VALID_SIZE, SEED, groups)
GTR = None if groups is None else groups.iloc[Xtr.index]   # grup untuk CV
print(Xtr.shape, Xva.shape)

## 4. Model yang akan dijelaskan
Penjelasan selalu dihitung pada **validation**, bukan training.

In [ ]:
from lightgbm import LGBMRegressor

model = LGBMRegressor(n_estimators=400, learning_rate=0.05, num_leaves=31, min_child_samples=10,
                      subsample=0.8, subsample_freq=1, colsample_bytree=0.8, random_state=42, n_jobs=-1, verbose=-1)
pipe = build_pipeline(model, NUM, CAT, kind='tree').fit(Xtr, ytr)

In [ ]:
pred = pipe.predict(Xva)
metrics = regression_metrics(yva, pred)
display(show_table(metrics))
plot_regression(yva, pred, path=OUT / "regression.png"); plt.show()

Matriks fitur setelah preprocessing (nama kolom dipertahankan). `mdl` = model di dalam pipeline.

In [ ]:
from superdb.tabular import feature_names
names = feature_names(pipe)
pre, mdl = pipe.named_steps["pre"], pipe.named_steps["model"]
Ztr = pd.DataFrame(pre.transform(Xtr), columns=names, index=Xtr.index)
Zva = pd.DataFrame(pre.transform(Xva), columns=names, index=Xva.index)
Zva.shape

## 5. Latih pohon dangkal untuk meniru model
Target pohon = **prediksi model** (bukan label asli).

In [ ]:
from sklearn.tree import DecisionTreeRegressor, export_text, plot_tree
from sklearn.metrics import r2_score
f = (lambda Z: mdl.predict_proba(Z)[:, 1]) if TASK == "classification" else mdl.predict
ftr, fva = f(Ztr), f(Zva)
rows = []
for d in (2, 3, 4, 5, 6):
    s = DecisionTreeRegressor(max_depth=d, min_samples_leaf=15, random_state=SEED).fit(Ztr, ftr)
    rows.append({"depth": d, "fidelity_R2_valid": r2_score(fva, s.predict(Zva))})
display(pd.DataFrame(rows).round(3))

## 6. Aturan yang bisa ditulis di laporan
Pilih kedalaman terkecil dengan fidelity memadai (mis. R² > 0.8).

In [ ]:
sur = DecisionTreeRegressor(max_depth=3, min_samples_leaf=15, random_state=SEED).fit(Ztr, ftr)
print(export_text(sur, feature_names=list(Ztr.columns), decimals=2))
fig, ax = plt.subplots(figsize=(13, 5)); plot_tree(sur, feature_names=list(Ztr.columns), filled=True, rounded=True, fontsize=8, ax=ax)
fig.savefig(OUT / "surrogate_tree.png", dpi=130, bbox_inches="tight"); plt.show()

In [ ]:
save_json({"valid": metrics, "method": "global_surrogate"}, OUT / "metrics.json")
save_model({"pipe": pipe, "cols": cols}, OUT / "model.joblib"); print("OK. hasil di", OUT)

## Langkah berikutnya
- Penjelasan model adalah bahan bab *analisis* di laporan: tulis 3 temuan utama + satu contoh kasus.
- Fitur yang 'terlalu penting' (mis. satu kolom mendominasi) = curigai kebocoran (lihat `notebooks/tabular/validation/`).